In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
read_query = (spark.readStream
              .format('delta')
              .table('bronze.raw_btcusdt_spot'))

In [0]:
cleaned_query = (
    read_query.drop('local_timestamp')
    .withColumns({
        'timestamp': timestamp_millis(col('timestamp').cast(LongType())),
        'id': col('Id').try_cast(LongType()),
        'price': col('price').try_cast(DecimalType(8, 3)),
        'quantity': col('quantity').try_cast(DecimalType(6, 5)),
        'side': col('side').try_cast(StringType())
    })
    .withColumns({
        'order_amount': col('quantity') * col('price'),
        'date': to_date(col('timestamp')),
        'hour': hour(col('timestamp')),
        'minute': minute(col('timestamp')),
        'second': second(col('timestamp'))
        })
    )

In [0]:
write_query = (cleaned_query.writeStream
               .format('delta')
               .outputMode('append')
               .trigger(availableNow=True)
               .option('checkpointLocation', "/Volumes/dbw_devfrancecentrallo1b/silver/silver_checkpoints")
               .table('silver.processed_btcusdt')
               )

In [0]:
write_query.awaitTermination()